In [ ]:
import os, sys, time, glob, site, shutil, subprocess, threading
from pathlib import Path

T0 = time.time()
BRANCH = "t4-kaggle-d12"
REPO = "https://github.com/michaelp91-dev/tinyforge.git"
MAX_STEPS = 2600
TOTAL_BATCH = 262144
B, T = 16, 512
TRAIN_TOKENS = 720_000_000
TOKENIZER_CHARS = 500_000_000
SESSION_HOURS = 11.5
CKPT_EVERY = 60
WORK = Path("/kaggle/working")
DATA = WORK / "data"
RUN = WORK / "run_d12"
SRC = Path("/tmp/tinyforge")

def sh(cmd, **kw):
    print("+", cmd if isinstance(cmd, str) else " ".join(map(str, cmd)), flush=True)
    subprocess.run(cmd, shell=isinstance(cmd, str), check=True, **kw)

NGPU = len([l for l in subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.splitlines() if l.startswith("GPU")]) if shutil.which("nvidia-smi") else 0
print("GPUs:", NGPU)
sh("nvidia-smi -L || true")
sh("free -g")
shutil.rmtree(SRC, ignore_errors=True)
SRC.parent.mkdir(parents=True, exist_ok=True)
sh(["git", "clone", "-q", "-b", BRANCH, REPO, str(SRC)])
sh([sys.executable, "-m", "pip", "install", "-q", "tokenizers", "transformers", "datasets", "pyarrow", "gguf", "safetensors"])
os.chdir(SRC)

In [ ]:
def find_nccl():
    roots = site.getsitepackages() + [site.getusersitepackages()]
    for r in roots:
        d = Path(r) / "nvidia" / "nccl"
        if (d / "include" / "nccl.h").exists():
            return d
    return None

if NGPU:
    os.environ["PATH"] = "/usr/local/cuda/bin:" + os.environ["PATH"]
    sh("nvcc --version | tail -2")
    flags = ["GPU=t4"]
    if NGPU > 1:
        nccl = find_nccl()
        if nccl is None:
            sh([sys.executable, "-m", "pip", "install", "-q", "nvidia-nccl-cu12"])
            nccl = find_nccl()
        flags += ["USE_NCCL=1", f"NCCL_HOME={nccl}"]
    sh(["make", "train_tinyforge"] + flags)
    sh("ls -l train_tinyforge")
else:
    print("No GPU in this session: this run only prepares the tokenizer and data.")

In [ ]:
def step_of(p):
    return int(p.name.split("_")[1])

for d in (DATA, RUN):
    d.mkdir(parents=True, exist_ok=True)
inp = Path("/kaggle/input")
if not (DATA / "tokenizer" / "tokenizer.json").exists():
    for p in inp.glob("**/data/tokenizer/tokenizer.json"):
        shutil.copytree(p.parent, DATA / "tokenizer", dirs_exist_ok=True)
        print("tokenizer from", p.parent)
        break
if not (DATA / "climbmix" / "val.bin").exists():
    for p in inp.glob("**/data/climbmix/val.bin"):
        shutil.copytree(p.parent, DATA / "climbmix", dirs_exist_ok=True)
        print("data from", p.parent)
        break
local = max([step_of(p) for p in RUN.glob("DONE_*")], default=0)
prev = sorted(inp.glob("**/run_d12/DONE_*"), key=step_of)
if prev and step_of(prev[-1]) > local:
    s = step_of(prev[-1])
    for name in (f"model_{s:08d}.bin", f"state_{s:08d}.bin", f"DONE_{s:08d}"):
        shutil.copy2(prev[-1].parent / name, RUN / name)
    if (prev[-1].parent.parent / "train.log").exists():
        shutil.copy2(prev[-1].parent.parent / "train.log", WORK / "train.log")
    local = s
    print("checkpoint from", prev[-1].parent, "step", s)
print(f"resume step {local} of {MAX_STEPS}")
sh("du -sh /kaggle/working/* || true")

In [ ]:
if not (DATA / "tokenizer" / "tokenizer.json").exists():
    sh([sys.executable, "python/train_tokenizer.py", "--out", str(DATA / "tokenizer"),
        "--vocab-size", "32768", "--max-chars", str(TOKENIZER_CHARS)])
sh([sys.executable, "python/prepare_climbmix.py", "--tokenizer", str(DATA / "tokenizer"),
    "--out", str(DATA / "climbmix"), "--target-tokens", str(TRAIN_TOKENS), "--val-tokens", "262144"])
sh(f"ls -lh {DATA}/climbmix")

In [ ]:
def train():
    budget = int(SESSION_HOURS * 3600 - (time.time() - T0) - 20 * 60)
    print(f"training time budget this session: {budget / 3600:.2f} h", flush=True)
    args = [str(SRC / "train_tinyforge"),
            "-i", f"{DATA}/climbmix/train_*.bin", "-j", f"{DATA}/climbmix/val.bin",
            "-e", "d12", "-o", str(RUN), "-b", str(B), "-t", str(T), "-d", str(TOTAL_BATCH),
            "-x", str(MAX_STEPS), "-v", "250", "-m", "20", "-n", str(CKPT_EVERY), "-y", "1",
            "--keep-ckpts", "2", "--time-limit", str(budget), "--vocab", "32768",
            "--token-bytes", f"{DATA}/tokenizer/token_bytes.bin"]
    env = dict(os.environ, NCCL_P2P_DISABLE="1", NCCL_IB_DISABLE="1")
    others = []
    if NGPU > 1:
        sync = Path("/tmp/tf_sync")
        shutil.rmtree(sync, ignore_errors=True)
        sync.mkdir()
        dist = ["--nproc", str(NGPU), "--sync-dir", str(sync)]
        for r in range(1, NGPU):
            others.append(subprocess.Popen(args + dist + ["--rank", str(r)], env=env,
                                           stdout=subprocess.DEVNULL, stderr=open(WORK / f"rank{r}.err", "w")))
        args = args + dist + ["--rank", "0"]
    p = subprocess.Popen(args, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

    def watchdog():
        while p.poll() is None:
            if any(o.poll() not in (None, 0) for o in others):
                print("another rank died; stopping rank 0", flush=True)
                p.kill()
            time.sleep(10)
    threading.Thread(target=watchdog, daemon=True).start()
    with open(WORK / "train.log", "a") as log:
        for line in p.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
    rc = p.wait()
    for o in others:
        try:
            o.wait(timeout=120)
        except subprocess.TimeoutExpired:
            o.kill()
    for r in range(1, NGPU):
        e = (WORK / f"rank{r}.err").read_text().strip()
        if e:
            print(f"rank {r} stderr:\n{e}")
    print("trainer exit code", rc)
    return rc

if NGPU:
    train()

In [ ]:
done = max([step_of(p) for p in RUN.glob("DONE_*")], default=0)
if done >= MAX_STEPS:
    ckpt = RUN / f"model_{done:08d}.bin"
    sh([sys.executable, "python/export_hf.py", "--input", str(ckpt), "--output", str(WORK / "d12-hf"),
        "--tokenizer", str(DATA / "tokenizer")])
    sh([sys.executable, "python/convert_tinyforge_to_gguf.py", "--model", str(WORK / "d12-hf"),
        "--out", str(WORK / "d12-base-f32.gguf")])
    print("FINISHED: d12-hf/ and d12-base-f32.gguf are in this version's Output.")
else:
    print(f"step {done} of {MAX_STEPS}. To continue: Save Version again with this version's output as an input.")
sh("du -sh /kaggle/working/* || true")